# SE3082 – Parallel Computing
## Lab Sheet 9 – Introduction to CUDA (Google Colab Edition)

**Student ID:** IT24610803  
**Module:** SE3082 Parallel Computing  
**Hardware Accelerator:** GPU (Tesla T4)  

---
### Objectives:
1. Switch on a GPU runtime in Colab and confirm visibility via `nvidia-smi`
2. Write, compile (`nvcc`), and run CUDA programs directly within notebook cells
3. Master the host/device memory pattern: **Copy in (Host -> Device) -> Launch Kernel -> Copy out (Device -> Host)**
4. Map threads and blocks across 1D vectors and 2D matrices with essential boundary guards

## Exercise 1 – Set up your CUDA environment in Google Colab

### Step 3: Check that the GPU is visible
Verify that Google Colab is connected to an NVIDIA GPU (e.g., Tesla T4) with ~15 GB VRAM.

In [ ]:
# Check GPU availability and details
!nvidia-smi

### Step 4: Check the CUDA compiler (`nvcc`)

In [ ]:
# Verify NVIDIA CUDA Compiler (nvcc) is installed
!nvcc --version

### Step 5: Check GPU Name and Compute Capability

In [ ]:
# Query GPU name and compute capability
!nvidia-smi --query-gpu=name,compute_cap --format=csv

### Step 7: Smoke test – Hello from the GPU (`hello.cu`)
We launch 1 block of 4 threads. Each thread prints its unique `threadIdx.x`.

In [ ]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();       // 1 block of 4 threads
    cudaDeviceSynchronize(); // wait so the GPU's output is flushed
    return 0;
}

In [ ]:
# Compile and run hello.cu
!nvcc -arch=native -o hello.o hello.cu && ./hello.o

--- 
## Exercise 2 – Simple calculation program from the lecture

Adds two integers on the GPU using one block containing one thread.
- Allocates memory on device using `cudaMalloc`
- Copies host inputs `a=2`, `b=7` to device using `cudaMemcpyHostToDevice`
- Launches `add<<<1, 1>>>` kernel
- Verifies kernel launch via `cudaGetLastError()`
- Copies result back to host using `cudaMemcpyDeviceToHost`

In [ ]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (the NVIDIA GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;          // host copies of a, b, c
    int *d_a, *d_b, *d_c; // device copies of a, b, c
    int size = sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Set up input values
    a = 2;
    b = 7;

    // 1. Copy the inputs from host to device
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // 2. Launch the add() kernel on the GPU: 1 block, 1 thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Colab edition: report a failed launch instead of silently printing garbage
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // 3. Copy the result from device to host
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run ex2_add.cu
!nvcc -arch=native -o ex2_add.o ex2_add.cu && ./ex2_add.o

### Exercise 2 – THINK Questions & Answers

1. **Why do we need separate variables `d_a`, `d_b`, `d_c` on the device? Why can the kernel not use `a`, `b`, `c` directly?**
   - *Answer:* The host (CPU) and device (GPU) possess physically discrete memory systems (System RAM vs. GPU VRAM). Pointers to CPU memory addresses are invalid and cannot be accessed directly by GPU threads. GPU kernel instructions require pointers pointing to memory explicitly allocated on device VRAM via `cudaMalloc`.

2. **In `cudaMemcpy(d_a, &a, ...)` we pass `&a`, but in the kernel launch we pass `d_a`. Why?**
   - *Answer:* In `cudaMemcpy`, `a` is a local CPU stack variable of type `int`, so `&a` (its memory address, type `int*`) is passed as the source pointer. `d_a` is already a pointer (type `int*`) holding the device address allocated by `cudaMalloc`. When launching the kernel `add<<<1, 1>>>(d_a, d_b, d_c)`, the kernel parameters expect device pointers (`int*`), so we pass `d_a` directly by value.

3. **How many threads run the `add` kernel? What would you change to run it with 10 threads, and would the answer change?**
   - *Answer:* Exactly 1 thread runs the kernel (specified by `<<<1, 1>>>`). To run with 10 threads, change the launch configuration to `add<<<1, 10>>>(d_a, d_b, d_c)`. The resulting numerical answer would remain `9` because each of the 10 threads would compute `*c = 2 + 7 = 9` and write to the same memory location, but it would cause redundant, concurrent writes.

--- 
## Exercise 3 – Vector addition using 512 blocks and one thread

Adds two vectors of 512 integers using **512 blocks of 1 thread each** (`<<<512, 1>>>`).
Inside the kernel, `blockIdx.x` is used to index elements.

In [ ]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;          // host copies of a, b, c
    int *d_a, *d_b, *d_c;    // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU with N blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run ex3_blocks.cu
!nvcc -arch=native -o ex3_blocks.o ex3_blocks.cu && ./ex3_blocks.o

### Exercise 3 – PREDICT BEFORE YOU RUN

- **How many blocks and how many threads in total does `add<<<N, 1>>>` launch?**  
  *Answer:* It launches **512 blocks**, with **1 thread per block**, totaling **512 threads**.

- **A GPU executes threads in groups of 32 called warps. What does that mean for a block that contains only one thread?**  
  *Answer:* NVIDIA GPUs schedule and execute instructions at the granularity of a **warp (32 threads)**. If a thread block has only 1 thread, the Streaming Multiprocessor (SM) still allocates a full 32-thread warp, but **31 threads remain disabled/idle**. This yields poor hardware SIMD efficiency (only $1/32 \approx 3.125\%$ active lane utilization).

--- 
## Exercise 4 – Vector addition using one block and 512 threads

Calculates vector addition using **1 block with 512 threads** (`<<<1, N>>>`).
Inside the kernel, `threadIdx.x` is used to index elements.

In [ ]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;          // host copies of a, b, c
    int *d_a, *d_b, *d_c;    // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run ex4_threads.cu
!nvcc -arch=native -o ex4_threads.o ex4_threads.cu && ./ex4_threads.o

### Exercise 4 – EXPERIMENT: Find the Limit
What happens when we test `#define N 2048` in a single block?

In [ ]:
%%writefile ex4_limit.cu
#include <cstdio>
#include <cstdlib>

#define N 2048  // Exceeds max threads per block limit (1024)

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Attempt launch with 2048 threads in 1 block
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run the limit test
!nvcc -arch=native -o ex4_limit.o ex4_limit.cu && ./ex4_limit.o

### Exercise 4 – Limit Experiment Observations
- **What is printed?**  
  `Kernel launch failed: invalid argument` (or `invalid configuration argument` depending on CUDA runtime version)
- **Which part of the program told you?**  
  `cudaGetLastError()` and `printf("Kernel launch failed: %s\n", cudaGetErrorString(err));`
- **Which limit of the GPU did you hit?**  
  Modern NVIDIA architectures have a hardware limit of **1024 threads per block** (`maxThreadsPerBlock = 1024`). Setting $N=2048$ exceeds this architectural limit, causing the CUDA runtime driver to reject the configuration.

--- 
## Exercise 5 – Multiply two vectors using blocks and threads (`ex5_vecmul.cu`)

### Implementation Details (All TODOs Solved):
- **Array size:** $N = 10,000,000$ elements
- **Threads per block:** $512$
- **TODO 1:** Global thread index: `int i = blockIdx.x * blockDim.x + threadIdx.x;`
- **TODO 2:** Bounds checking: `if (i < n) { c[i] = a[i] * b[i]; }`
- **TODO 3:** Number of blocks rounded up: `int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;` ($19,532$ blocks)
- **TODO 4:** Kernel invocation: `vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);`
- **TODO 5:** Result copied to host: `cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);`
- **TODO 6:** Print last 1,000 elements ($i = 9,999,000$ to $9,999,999$)
- **TODO 7:** CPU validation loop checking all $10,000,000$ elements

In [ ]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000 // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread (use blockIdx, blockDim, threadIdx)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array. Why is this needed?
    // With 19,532 blocks * 512 threads = 10,000,384 threads,
    // the last 384 threads would access out-of-bounds memory without this guard.
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;          // host copies of a, b, c
    int *d_a, *d_b, *d_c;    // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: print the last 1000 results as "index) a x b = c"
    for (int r = N - 1000; r < N; r++) {
        printf("%d) %d x %d = %d\n", r, a[r], b[r], c[r]);
    }

    // TODO 7: verify all N results on the CPU and print the number of mismatches
    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] * b[r]) errors++;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run ex5_vecmul.cu
!nvcc -arch=native -o ex5_vecmul.o ex5_vecmul.cu && ./ex5_vecmul.o

### Exercise 5 – THINK Question & Self-Check

**Question:** Why is it a good idea to use a multiple of 32 for the threads per block? What happens to the total number of threads launched if you use 500 instead of 512?
- *Answer:* CUDA hardware executes threads in units of **warps (32 threads)**. If a block size is not a multiple of 32 (like 500), the final warp contains only $500 \pmod{32} = 20$ active threads while 12 threads remain idle, reducing warp execution efficiency. Furthermore, for $N = 10,000,000$ elements, 500 threads/block requires $20,000$ blocks, wasting $20,000 \times 12 = 240,000$ warp lanes across execution.

| Metric | Expected Value |
| :--- | :--- |
| **Blocks launched** | `19532 blocks × 512 threads` |
| **Total threads launched** | `10,000,384` |
| **Active threads in last block**| `128` (remaining 384 stopped by guard) |
| **First of last 1000** | `9999000) 76 x 52 = 3952` |
| **Last result** | `9999999) 31 x 73 = 2263` |
| **Verification** | `0 mismatches out of 10000000` |

--- 
## Exercise 6 – Multiply two 10,000 × 10,000 matrices element by element (2D Grid)

### Implementation Details (All TODOs Solved):
- Matrix dimension: $10,000 \times 10,000 = 100,000,000$ elements ($400\text{ MB}$ per matrix, $1.2\text{ GB}$ total per space).
- Block dimension: `dim3 threadsPerBlock(32, 16);` ($32 \times 16 = 512$ threads/block)
- **TODO 1:** `col = blockIdx.x * blockDim.x + threadIdx.x;` and `row = blockIdx.y * blockDim.y + threadIdx.y;`
- **TODO 2 & 3:** Guard both coordinates and calculate row-major flattened index:  
  `if (row < rows && col < cols) { int idx = row * cols + col; c[idx] = a[idx] * b[idx]; }`
- **TODO 4:** Round up blocks in both dimensions:  
  `dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x, (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);` ($313 \times 625 = 195,625$ blocks)
- **TODO 5:** Kernel invocation: `mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);`
- **TODO 6:** Result copied to host: `cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);`
- **TODO 7:** Print last 1,000 elements: row $9999$, columns $9000$ to $9999$
- **TODO 8:** CPU validation checking all $100,000,000$ elements

In [ ]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c] (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    // TODO 1: row comes from the y dimension, col from the x dimension
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // TODO 2: only compute if BOTH row and col are inside the matrix
    // TODO 3: convert (row, col) into a flat index: row * cols + col
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;          // host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;    // device copies
    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int); // 400 MB per matrix

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);

    // TODO 4: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    // TODO 5: launch mulElem with <<<numBlocks, threadsPerBlock>>>
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 6: copy d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: print C[9999][9000] ... C[9999][9999] (the last 1000 elements)
    for (int col = 9000; col < COLS; col++) {
        int idx = 9999 * COLS + col;
        printf("C[9999][%d] %d x %d = %d\n", col, a[idx], b[idx], c[idx]);
    }

    // TODO 8: verify all elements on the CPU and print the number of mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) errors++;
    }
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

In [ ]:
# Compile and run ex6_matrix.cu
!nvcc -arch=native -o ex6_matrix.o ex6_matrix.cu && ./ex6_matrix.o

### Exercise 6 – Self-Check Validation

| Item | Expected value |
| :--- | :--- |
| **Grid** | `Grid: 313 x 625 blocks (195,625 blocks)` |
| **Total threads launched** | `100,160,000` ($160,000$ idle threads beyond $100,000,000$ protected by guards) |
| **C[9999][9000]** | `29 x 56 = 1624` |
| **C[9999][9999]** | `86 x 21 = 1806` |
| **Verification** | `0 mismatches out of 100000000` |